# BHJet original versus OpenMP comparison

This notebook compares the original `docs` checkout against the `experiment/openmp-zones` worktree. Each case runs in a separate Python process so the two native modules named `bhjet` cannot collide.

It checks numerical equality for the total spectrum and all emission components before reporting any speed-up. The OpenMP one-thread and four-thread timings use the same Release-built extension, so their speed-up isolates parallelisation.

In [ ]:
from pathlib import Path
import os


def find_project_root(start):
    for directory in (start, *start.parents):
        if (directory / 'python' / 'bhjet').is_dir() and (directory / 'CMakeLists.txt').is_file():
            return directory
    raise RuntimeError('Run this notebook from inside the OpenMP worktree.')


CURRENT_ROOT = find_project_root(Path.cwd().resolve())
OPENMP_ROOT = Path(os.environ.get('BHJET_OPENMP_ROOT', CURRENT_ROOT)).resolve()
ORIGINAL_ROOT = Path(
    os.environ.get('BHJET_ORIGINAL_ROOT', OPENMP_ROOT.parent / 'modBHJet')
).resolve()

assert OPENMP_ROOT != ORIGINAL_ROOT

for label, root in {'original': ORIGINAL_ROOT, 'openmp': OPENMP_ROOT}.items():
    extension = list((root / 'python' / 'bhjet').glob('bhjet*.so'))
    assert extension, f'No local extension found for {label}: {root}'
    print(f'{label:8s}: {root}')
    print(f'          {extension[0]}')

## Run isolated model evaluations

The original code is run once. The OpenMP branch is run with one and four threads. All three calculations use the same default model parameters and the same 201-point energy grid.

In [ ]:
import json
import subprocess
import sys

import numpy as np


WORKER_CODE = r'''
import json
import os
from pathlib import Path
import subprocess
import time

import astropy.units as u
import numpy as np
import bhjet
from bhjet.gammapy import BHJetSpectralModel

root = Path(os.environ['BHJET_ROOT'])
energy = np.geomspace(1e-8, 1e12, 201) * u.eV
model = BHJetSpectralModel()

start = time.perf_counter()
total = model(energy)
elapsed = time.perf_counter() - start
components = model.evaluate_components(energy)

print(json.dumps({
    'package_path': bhjet.__file__,
    'commit': subprocess.check_output(
        ['git', '-C', str(root), 'rev-parse', '--short', 'HEAD'], text=True
    ).strip(),
    'elapsed_s': elapsed,
    'energy_eV': energy.to_value(u.eV).tolist(),
    'dnde_total': total.to_value('cm-2 s-1 erg-1').tolist(),
    'components': {
        name: values.to_value('cm-2 s-1').tolist()
        for name, values in components.items()
    },
}))
'''


def run_case(label, root, threads):
    environment = os.environ.copy()
    environment['BHJET_ROOT'] = str(root)
    environment['OMP_NUM_THREADS'] = str(threads)
    environment['PYTHONPATH'] = str(root / 'python') + os.pathsep + environment.get('PYTHONPATH', '')

    result = subprocess.run(
        [sys.executable, '-c', WORKER_CODE],
        env=environment,
        check=True,
        capture_output=True,
        text=True,
    )
    payload = json.loads(result.stdout.splitlines()[-1])
    assert str(root) in payload['package_path'], f'{label} imported the wrong package'
    return payload


original = run_case('original', ORIGINAL_ROOT, threads=1)
openmp_one = run_case('openmp-one', OPENMP_ROOT, threads=1)
openmp_four = run_case('openmp-four', OPENMP_ROOT, threads=4)

for label, result in {
    'original': original,
    'openmp (1 thread)': openmp_one,
    'openmp (4 threads)': openmp_four,
}.items():
    print(f"{label:18s} commit={result['commit']}  elapsed={result['elapsed_s']:.3f} s")
    print(f"{'':18s} package={result['package_path']}")

In [ ]:
expected_components = {'pre_syn', 'pre_com', 'post_syn', 'post_com', 'total'}
assert set(original['components']) == expected_components

comparisons = [
    ('original', original, 'openmp (1 thread)', openmp_one),
    ('openmp (1 thread)', openmp_one, 'openmp (4 threads)', openmp_four),
]

for left_label, left, right_label, right in comparisons:
    np.testing.assert_allclose(left['energy_eV'], right['energy_eV'], rtol=0.0, atol=0.0)
    np.testing.assert_allclose(left['dnde_total'], right['dnde_total'], rtol=1e-12, atol=0.0)

    for component in expected_components:
        np.testing.assert_allclose(
            left['components'][component],
            right['components'][component],
            rtol=1e-12,
            atol=0.0,
        )

    print(f'{left_label} and {right_label} agree for the total and all components.')

## Spectral and runtime comparison

The plot uses the total spectrum in the common `E² dN/dE` representation. The ratio should remain at one within floating-point precision. The OpenMP speed-up is calculated between one and four threads from the same build.

In [ ]:
import matplotlib.pyplot as plt

energy_eV = np.asarray(original['energy_eV'])
energy_erg = energy_eV * 1.602176634e-12

original_sed = energy_erg**2 * np.asarray(original['dnde_total'])
openmp_sed = energy_erg**2 * np.asarray(openmp_four['dnde_total'])
ratio = openmp_sed / original_sed

fig, (ax_spectrum, ax_ratio, ax_time) = plt.subplots(
    3,
    1,
    figsize=(8, 10),
    gridspec_kw={'height_ratios': [3, 1, 1]},
)

ax_spectrum.loglog(energy_eV, original_sed, label='original', color='tab:blue')
ax_spectrum.loglog(energy_eV, openmp_sed, '--', label='OpenMP (4 threads)', color='tab:orange')
ax_spectrum.set_ylabel(r'$E^2 dN/dE$ [erg cm$^{-2}$ s$^{-1}$]')
ax_spectrum.legend()

ax_ratio.semilogx(energy_eV, ratio, color='black')
ax_ratio.axhline(1.0, color='tab:red', linestyle='--')
ax_ratio.set_ylabel('OpenMP / original')
ax_ratio.set_xlabel('Energy [eV]')

labels = ['original', 'OpenMP\n1 thread', 'OpenMP\n4 threads']
times = [original['elapsed_s'], openmp_one['elapsed_s'], openmp_four['elapsed_s']]
ax_time.bar(labels, times, color=['tab:blue', 'tab:orange', 'tab:green'])
ax_time.set_ylabel('Solve time [s]')

speedup = openmp_one['elapsed_s'] / openmp_four['elapsed_s']
ax_time.set_title(f'OpenMP 1-thread to 4-thread speed-up: {speedup:.2f}x')
fig.tight_layout()
plt.show()